# 현금영수증 한 건의 전체 정보 조회

1. 0번 설정에서 `SDATE`를 실제 발급일(YYYYMMDD)로, `AUTHNO`를 영수증 승인번호로 바꾸세요. 번호는 따옴표 안에 입력해 앞자리 0을 유지합니다.
2. 위에서부터 실행한 뒤 4번 조회 결과를 확인하세요. 승인번호를 모르면 빈 값으로 두고 해당 날짜의 목록에서 한 건을 고를 수 있습니다.
3. 4-1번의 `ROW_INDEX`를 목록의 행 번호로 설정하고 실행하면 해당 거래의 **반환된 모든 필드**, **명세 대비 누락 필드**, **원본 JSON**을 표시합니다. 기본값 0은 첫 번째 행입니다.

조회는 한 페이지씩 수행합니다. 한 승인번호에 승인·취소 등 여러 행이 있으면 목록에서 원하는 행을 선택하세요. 다음 페이지가 필요하면 `CURRPAGE`를 변경하고 4번부터 다시 실행합니다.
API가 제공하지 않는 정보나 마스킹된 원문을 추가로 복원하는 기능은 아닙니다.

필요 패키지: `pandas` (요청은 `van_test_helpers.py`가 표준 라이브러리로 처리).
저장소 루트의 `.env`에 `SMARTRO_VAN_API_KEY`와 `SMARTRO_VAN_TERMID`(단말기번호 10자리)를 설정하세요. 인증키를 노트북에 직접 적지 마세요.
GET 조회만 수행하며 발급·취소 요청은 하지 않습니다. 실제 거래 API 호출 없이 검증했습니다.

공개 명세: https://exttran.smilebiz.co.kr/getApiSvcInfoData?SVC_CTGR=VAN

## 0. 공통 설정 (Bearer 인증 + 요청 함수)

In [ ]:
import json
import sys
from pathlib import Path
from datetime import datetime

import pandas as pd
from IPython.display import display

# 노트북 폴더 또는 저장소 루트 어디에서 실행하든 헬퍼를 찾는다.
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = [parent, parent / "smilebiz-van/van-api"]
    helper_dir = next((p for p in candidates if (p / "van_test_helpers.py").exists()), None)
    if helper_dir is not None:
        sys.path.insert(0, str(helper_dir))
        break
else:
    raise RuntimeError("노트북 폴더 또는 저장소 루트에서 실행하세요.")
from van_test_helpers import van_get, rows_of, env_value

TERMID = env_value("SMARTRO_VAN_TERMID")   # 조회할 단말기번호 10자리 (.env)
SDATE = "20260901"   # 실제 발급일(YYYYMMDD)로 변경
EDATE = "20260907"
AUTHNO = ""          # 조회할 영수증 승인번호. 모르면 빈 값으로 두고 목록에서 선택
GBN = "2"
CURRPAGE = 0

STIME = ""
ETIME = ""
COMP_NO = ""
COMP_IDX = ""


def validate_filters():
    if len(TERMID) != 10:
        raise ValueError("TERMID는 10자리 문자열이어야 합니다. .env의 SMARTRO_VAN_TERMID를 확인하세요.")
    for value in (SDATE, EDATE):
        if len(value) != 8 or not value.isdigit():
            raise ValueError("날짜는 YYYYMMDD 8자리여야 합니다.")
        datetime.strptime(value, "%Y%m%d")
    if SDATE > EDATE:
        raise ValueError("시작일이 종료일보다 늦습니다.")
    for value in (STIME, ETIME):
        if value:
            if len(value) != 6 or not value.isdigit():
                raise ValueError("시간은 hhmmss 6자리 또는 빈 값이어야 합니다.")
            datetime.strptime(value, "%H%M%S")
    if SDATE == EDATE and STIME and ETIME and STIME > ETIME:
        raise ValueError("시작시간이 종료시간보다 늦습니다.")
    if GBN != "2":
        raise ValueError("이 노트북은 현금영수증 GBN=2 전용입니다.")
    if type(CURRPAGE) is not int or CURRPAGE < 0:
        raise ValueError("페이지는 0 이상의 정수여야 합니다.")


validate_filters()
print(f"단말기: {TERMID} | 현금영수증 | 기간: {SDATE} ~ {EDATE}")
print("인증키 설정됨 (값은 표시하지 않음)")

## 1. 서버연결 상태체크
성공해도 해당 가맹점의 매출 조회 권한까지 확인된 것은 아닙니다.

In [ ]:
check = van_get("/V1/common/serverChecks")
check

## 2. 공통코드정보조회

`GET /V1/common/getCommCodeInfo` — 파라미터 없음. `GBN`(승인구분), `HID_GBN`(카드사구분),
`ORGCOD`(카드사코드) 등 다른 API 호출 시 필요한 코드값을 확인할 수 있습니다.

In [ ]:
comm_codes = van_get("/V1/common/getCommCodeInfo")
cash_codes = [c for g in comm_codes.get("CODE_INFO", []) if g.get("GROUP_CODE") == "GBN"
              for c in g.get("CODES", []) if str(c.get("CODE")) == GBN]
print("현금영수증 코드 확인:", cash_codes)
if not cash_codes:
    raise RuntimeError("공통코드에서 GBN=2를 찾지 못했습니다. 스마트로에 확인하세요.")

## 3. 해당 단말기의 현금영수증 집계
집계 API에는 GBN 요청 필드가 없으므로 해당 단말기의 집계를 받은 뒤 현금영수증 행만 표시합니다.

In [ ]:
validate_filters()
sales_sum = van_get("/V1/sales/getSalesSum", params={
    "SDATE": SDATE, "EDATE": EDATE, "COMP_NO": COMP_NO, "COMP_IDX": COMP_IDX, "TERMID": TERMID,
})
sales_sum_df = pd.DataFrame([r for r in rows_of(sales_sum) if str(r.get("GBN")) == GBN])
print(f"현금영수증 집계 행 수: {len(sales_sum_df)}")
display(sales_sum_df)

## 4. 현금영수증 승인·취소 내역 (한 페이지)
설정한 단말기(`SMARTRO_VAN_TERMID`), GBN=2로 조회합니다. 날짜·승인번호·페이지는 0번 설정에서 바꾼 후 다시 실행하세요.

기존 노트북에는 서버가 빈 선택 필드도 요구한다는 테스트 기록이 있으므로 모든 요청 필드를 유지합니다.
페이지는 공개 명세에 따라 0부터 시작합니다. 반환되는 CURRPAGE/TOTPAGE를 확인하고 다음 페이지는 직접 지정하세요.
이 표는 전체 기간의 모든 페이지를 합친 결과가 아닙니다. 오류는 0건으로 처리하지 않고 중단합니다.

In [ ]:
# 재조회 실패 시 이전 결과를 상세 셀에서 사용하지 않도록 초기화합니다.
sales_list = None
rows = []
sales_list_df = pd.DataFrame()
validate_filters()
params = {
    "SDATE": SDATE, "STIME": STIME, "EDATE": EDATE, "ETIME": ETIME,
    "COMP_NO": COMP_NO, "COMP_IDX": COMP_IDX, "TERMID": TERMID,
    "GBN": GBN, "CDNO": "", "AUTHNO": AUTHNO, "HID_GBN": "", "ORGCOD": "",
    "REJEC_TYPE": "1",  # 거절도 포함하여 성공/실패를 구분
    "CURRPAGE": CURRPAGE,
}
sales_list = van_get("/V1/sales/getSalesList", params=params)
rows = rows_of(sales_list)
# 다른 단말기나 결제수단이 반환되면 결과를 신뢰하고 계속 진행하지 않습니다.
for row in rows:
    if row.get("TERMID") not in (None, "") and str(row["TERMID"]).strip() != TERMID:
        raise RuntimeError("요청과 다른 단말기 거래가 반환되었습니다. 조회 권한/필터를 확인하세요.")
    if row.get("GBN") not in (None, "") and str(row["GBN"]) != GBN:
        raise RuntimeError("현금영수증 외 거래가 반환되었습니다. 응답을 확인하세요.")
    if AUTHNO and str(row.get("AUTHNO", "" )).strip() != AUTHNO.strip():
        raise RuntimeError("요청과 다른 승인번호가 반환되었습니다. 응답 필터를 확인하세요.")
sales_list_df = pd.DataFrame(rows)
print(f"단말기 {TERMID} | {SDATE} ~ {EDATE} | 요청 페이지 {CURRPAGE} | 수신 {len(rows)}건")
for field in ("CURRPAGE", "TOTPAGE", "TOT_CNT", "TOT_AMT"):
    value = sales_list.get(field)
    if value is None and rows:
        value = rows[0].get(field)
    print(f"{field}: {value if value is not None else '응답에 없음'}")
if not rows:
    print("정상 응답이지만 이 페이지에 내역이 없습니다. 발급일·페이지·거래 반영 여부를 확인하세요.")
else:
    columns = {"TSDATE": "승인일자", "TSTIME": "승인시간", "AUTHNO": "승인번호",
               "NRSPC_NM": "승인/취소/거절", "AMT1": "금액", "AMT2": "봉사료", "AMT3": "부가세",
               "TERMID": "단말기번호", "ORGDATE": "원거래일자", "MTRCNO": "일련번호",
               "NRSPC_MSG": "승인결과", "GBN": "결제수단"}
    display(sales_list_df.reindex(columns=list(columns)).rename(columns=columns))

## 4-1. 현금영수증 한 건의 모든 정보
목록의 행 번호를 선택합니다. 조회 결과가 여러 건이어도 여기서는 한 행의 모든 응답 필드를 펼쳐 봅니다.

In [ ]:
# 4번 목록에 표시된 행 번호(index)를 입력하세요. 0은 첫 번째 거래입니다.
ROW_INDEX = 0

selected_receipt = None
receipt_detail_df = pd.DataFrame()
if sales_list is None or sales_list_df.empty:
    raise RuntimeError("먼저 4번 조회를 성공적으로 실행하세요. 0건이면 발급일·승인번호·페이지를 확인하세요.")
if type(ROW_INDEX) is not int or not 0 <= ROW_INDEX < len(sales_list_df):
    raise ValueError(f"ROW_INDEX는 0 ~ {len(sales_list_df) - 1} 사이 정수여야 합니다.")

# DataFrame으로 변환하기 전의 원본을 사용해 null과 숫자 등 원래 값을 유지합니다.
selected_receipt = rows_of(sales_list)[ROW_INDEX]
field_labels = {
    "TSDATE": "승인일자", "TSTIME": "승인시간",
    "COMP_NO": "사업자번호", "COMP_IDX": "사업자 IDX", "MEMBNAM": "가맹점명",
    "NRSPC_NM": "승인 / 취소 / 승인거절 / 취소거절",
    "CDNO": "마스킹된 번호 (명세상 카드번호)", "ISTMMON": "할부기간",
    "AMT1": "금액", "AMT2": "봉사료", "AMT3": "부가가치세", "AUTHNO": "승인번호",
    "HID": "발급사코드", "HID_NM": "발급사명",
    "ACQHID": "매입사코드", "ACQHID_NM": "매입사명",
    "DEPDATE": "입금예정일자", "TSFEE": "수수료", "DPAMT": "입금예정액",
    "TERMID": "단말기번호", "CHECK_FLAG": "체크카드여부",
    "ORGDATE": "원거래일자", "MTRCNO": "일련번호",
    "NRSPC_MSG": "승인결과코드/메시지", "HCODE_MSG": "반송응답코드/메시지",
    "ACQBANK": "간편결제명", "TR_TYPE": "카드거래타입 (MS / IC)", "GBN": "결제수단 코드",
    "CURRPAGE": "현재 페이지", "TOTPAGE": "전체 페이지 수 (명세 표기)",
    "TOT_CNT": "조회 조건의 총 건수 (한 건의 값 아님)",
    "TOT_AMT": "조회 조건의 총 금액 (한 건의 값 아님)",
}

def value_state(value):
    if value is None:
        return "null"
    if value == "":
        return "빈 문자열"
    return "값 있음"

# 알 수 없는 추가 필드도 제외하지 않고 전부 표시합니다. 0은 빈 값이 아닙니다.
receipt_detail_df = pd.DataFrame([
    {"필드": key, "설명": field_labels.get(key, "추가 응답 필드 (명세 설명 없음)"),
     "값(JSON 표기)": json.dumps(value, ensure_ascii=False), "상태": value_state(value)}
    for key, value in selected_receipt.items()
])
print(f"선택한 행: {ROW_INDEX} | 승인번호: {selected_receipt.get('AUTHNO', '없음')} | "
      f"구분: {selected_receipt.get('NRSPC_NM', '없음')} | 반환 필드: {len(selected_receipt)}개")
print("이 상세 표는 선택한 한 행입니다. 승인·취소 등 관련 행 전체를 합친 결과는 아닙니다.")
with pd.option_context("display.max_rows", None, "display.max_columns", None,
                       "display.max_colwidth", None):
    display(receipt_detail_df)

missing_fields = [key for key in field_labels if key not in selected_receipt and key not in sales_list]
print("공개 명세 필드 중 응답에 없는 항목:", ", ".join(missing_fields) or "없음")
print("카드 공통 필드는 현금영수증에서 빈 값이거나 생략될 수 있습니다.")
print("\n[선택한 현금영수증 한 행 — 원본 JSON]")
print(json.dumps(selected_receipt, ensure_ascii=False, indent=2))
print("\n[응답 공통 정보 — DATA 제외]")
print(json.dumps({key: value for key, value in sales_list.items() if key != "DATA"},
                 ensure_ascii=False, indent=2))
# 전체 페이지 원본은 sales_list, 선택한 거래 원본은 selected_receipt 변수에서 확인합니다.

## 5. (참고) 입금 관련 API

가이드 사이트(VAN 탭)에 아래 API가 더 있지만, 이 노트북에서는 요청 파라미터를 확인하지 않았습니다.
실제로 쓰시려면 https://exttran.smilebiz.co.kr 에서 `VAN` 탭 -> 해당 항목을 눌러 `Request` 표를 직접 확인한 뒤,
위 `van_get()` 함수에 경로와 파라미터만 맞춰서 그대로 재사용하시면 됩니다.

- 입금내역 조회(집계)
- 입금내역 조회(상세)
- 입금보류내역 조회
- 청구내역 조회